In [12]:
# ==========================
# 1. Ligação ao DuckDB
# ==========================

import duckdb

DB_PATH = "/Users/rr/Library/Mobile Documents/com~apple~CloudDocs/05.Salvesen/00_DB/2026.duckdb"

con = duckdb.connect(DB_PATH)


# ==========================
# 2. Análise por transportista
# ==========================

df = con.sql("""
    SELECT
        Transportista,
        COUNT(DISTINCT CODEUT) AS NUM_CODEUT,
        SUM(TRY_CAST(PALETS AS DOUBLE)) AS PALETES,
        SUM(TRY_CAST(COSTEDT AS DOUBLE)) AS COSTDT,
        SUM(TRY_CAST(INGRESODT AS DOUBLE)) AS INGRESO_DT
    FROM inform_27_2026
    WHERE GESTION = 'LIS'
    GROUP BY Transportista
    ORDER BY NUM_CODEUT DESC
""").df()

df

,TRANSPORTISTA,NUM_CODEUT,PALETES,COSTDT,INGRESO_DT
0,"TRANSPORTES FLORENCIO SILVA, LDA. -",5547,104310.941,1201183.79,682009.07
1,JMR PRESTAÇAO DE SERVIÇOS PARA A DISTRIBUÇAO S.A,1574,55789.250,408852.81,350167.60
2,MODELO CONTINENTE HIPERMERCADOS S.A.,1079,34895.730,255319.82,246887.41
3,Transportes Fernando Simões Monteiro Unip. Lda,1036,18761.000,227875.67,56663.51
4,CMTIR TRANSPORTES NAC. INTERN. S.A,939,9849.470,261983.96,141721.95
5,TRANSAURA TRANSPORTES LDA,671,18419.000,112486.05,134107.85
6,"TJA-TRANSPORTES J.AMARAL, S.A. -",596,14168.420,185865.23,176029.33
7,TRANSPORTES PAULO COSTA & FERREIRA LDA,386,12178.000,179557.34,163501.04
8,"TRANSPARENTODISSEIA - Transportes Unipessoal, ...",344,9249.000,114510.51,137158.27
9,WORLD RUNNERS UNIPESSOAL LDA,335,8798.700,75523.87,67714.98


In [1]:
import duckdb

DB_PATH_2025 = "/Users/rr/Library/Mobile Documents/com~apple~CloudDocs/05.Salvesen/00_DB/2025.duckdb"

con_2025 = duckdb.connect(DB_PATH_2025)

In [1]:
# ==========================
# 1. Configuração
# ==========================

from pathlib import Path
import sqlite3
import duckdb
import pandas as pd

BASE = Path.home() / "Library/Mobile Documents/com~apple~CloudDocs/05.Salvesen/00_DB"

BASES = {
    2025: {
        "sqlite": BASE / "2025.db",
        "duckdb": BASE / "2025.duckdb",
    },
    2026: {
        "sqlite": BASE / "2026.db",
        "duckdb": BASE / "2026.duckdb",
    },
}

In [2]:
# ==========================
# 2. Funções
# ==========================

def tabelas_sqlite(con):
    return {
        r[0]
        for r in con.execute("""
            SELECT name
            FROM sqlite_master
            WHERE type = 'table'
              AND name NOT LIKE 'sqlite_%'
        """).fetchall()
    }


def tabelas_duckdb(con):
    return {
        r[0]
        for r in con.execute("""
            SELECT table_name
            FROM information_schema.tables
            WHERE table_schema = 'main'
              AND table_type = 'BASE TABLE'
        """).fetchall()
    }


def colunas_sqlite(con, tabela):
    return {
        r[1]
        for r in con.execute(f'PRAGMA table_info("{tabela}")').fetchall()
    }


def colunas_duckdb(con, tabela):
    return {
        r[1]
        for r in con.execute(f'PRAGMA table_info("{tabela}")').fetchall()
    }

In [5]:
# ==========================
# 1. Configuração
# ==========================

from pathlib import Path
import sqlite3
import duckdb
import pandas as pd

BASE = Path.home() / "Library/Mobile Documents/com~apple~CloudDocs/05.Salvesen/00_DB"

BASES = {
    2025: {
        "sqlite": BASE / "2025.db",
        "duckdb": BASE / "2025.duckdb",
        "tabela": "inform_27_2025",
    },
    2026: {
        "sqlite": BASE / "2026.db",
        "duckdb": BASE / "2026.duckdb",
        "tabela": "inform_27_2026",
    },
}


# ==========================
# 2. Conversão numérica
# ==========================

def valor_sqlite(coluna):
    return f"""
        CAST(
            REPLACE(
                REPLACE(TRIM("{coluna}"), '.', ''),
                ',', '.'
            ) AS REAL
        )
    """


def valor_duckdb(coluna):
    return f"""
        TRY_CAST(
            REPLACE(
                REPLACE(TRIM("{coluna}"), '.', ''),
                ',', '.'
            ) AS DOUBLE
        )
    """


# ==========================
# 3. Validar
# ==========================

resultado = []

for ano, cfg in BASES.items():

    sql = sqlite3.connect(cfg["sqlite"])
    ddb = duckdb.connect(str(cfg["duckdb"]), read_only=True)

    tabela = cfg["tabela"]

    linhas_sql = sql.execute(
        f'SELECT COUNT(*) FROM "{tabela}"'
    ).fetchone()[0]

    linhas_ddb = ddb.execute(
        f'SELECT COUNT(*) FROM "{tabela}"'
    ).fetchone()[0]

    ingresso_sql = sql.execute(
        f'SELECT SUM({valor_sqlite("INGRESODT")}) FROM "{tabela}"'
    ).fetchone()[0]

    ingresso_ddb = ddb.execute(
        f'SELECT SUM({valor_duckdb("INGRESODT")}) FROM "{tabela}"'
    ).fetchone()[0]

    coste_sql = sql.execute(
        f'SELECT SUM({valor_sqlite("COSTEDT")}) FROM "{tabela}"'
    ).fetchone()[0]

    coste_ddb = ddb.execute(
        f'SELECT SUM({valor_duckdb("COSTEDT")}) FROM "{tabela}"'
    ).fetchone()[0]

    resultado.append({
        "Ano": ano,
        "Linhas_SQLite": linhas_sql,
        "Linhas_DuckDB": linhas_ddb,
        "Dif_Linhas": linhas_ddb - linhas_sql,
        "INGRESODT_SQLite": ingresso_sql,
        "INGRESODT_DuckDB": ingresso_ddb,
        "Dif_INGRESODT": ingresso_ddb - ingresso_sql,
        "COSTEDT_SQLite": coste_sql,
        "COSTEDT_DuckDB": coste_ddb,
        "Dif_COSTEDT": coste_ddb - coste_sql,
    })

    sql.close()
    ddb.close()


# ==========================
# 4. Resultado
# ==========================

df = pd.DataFrame(resultado)

df

,Ano,Linhas_SQLite,Linhas_DuckDB,Dif_Linhas,INGRESODT_SQLite,INGRESODT_DuckDB,Dif_INGRESODT,COSTEDT_SQLite,COSTEDT_DuckDB,Dif_COSTEDT
0,2025,1398519,1398519,0,2.732095e+09,2.732095e+09,0.0,6.048135e+09,6.048135e+09,0.0
1,2026,774239,774239,0,1.798828e+09,1.798828e+09,0.0,3.579670e+09,3.579670e+09,0.0


In [8]:
# ==========================
# 1. Configuração
# ==========================

from pathlib import Path
import sqlite3
import duckdb
import pandas as pd

BASE = Path.home() / "Library/Mobile Documents/com~apple~CloudDocs/05.Salvesen/00_DB"

BASES = {
    2025: {
        "sqlite": BASE / "2025.db",
        "duckdb": BASE / "2025.duckdb",
        "tabela": "inform_27_2025",
    },
    2026: {
        "sqlite": BASE / "2026.db",
        "duckdb": BASE / "2026.duckdb",
        "tabela": "inform_27_2026",
    },
}


# ==========================
# 2. Comparar conteúdo
# ==========================

resultado = []

for ano, cfg in BASES.items():

    sql = sqlite3.connect(cfg["sqlite"])
    ddb = duckdb.connect(str(cfg["duckdb"]), read_only=True)

    tabela = cfg["tabela"]

    df_sql = pd.read_sql_query(
        f'SELECT * FROM "{tabela}"',
        sql
    )

    df_ddb = ddb.execute(
        f'SELECT * FROM "{tabela}"'
    ).df()

    mesmas_colunas = list(df_sql.columns) == list(df_ddb.columns)

    df_sql = df_sql.astype("string").fillna("<NULL>")
    df_ddb = df_ddb.astype("string").fillna("<NULL>")

    hash_sql = pd.util.hash_pandas_object(
        df_sql,
        index=False
    ).sort_values().reset_index(drop=True)

    hash_ddb = pd.util.hash_pandas_object(
        df_ddb,
        index=False
    ).sort_values().reset_index(drop=True)

    conteudo_igual = hash_sql.equals(hash_ddb)

    resultado.append({
        "Ano": ano,
        "Linhas_SQLite": len(df_sql),
        "Linhas_DuckDB": len(df_ddb),
        "Colunas_SQLite": len(df_sql.columns),
        "Colunas_DuckDB": len(df_ddb.columns),
        "Mesmas_Colunas": mesmas_colunas,
        "Conteudo_Igual": conteudo_igual,
    })

    sql.close()
    ddb.close()


# ==========================
# 3. Resultado
# ==========================

df_validacao = pd.DataFrame(resultado)

df_validacao

,Ano,Linhas_SQLite,Linhas_DuckDB,Colunas_SQLite,Colunas_DuckDB,Mesmas_Colunas,Conteudo_Igual
0,2025,1398519,1398519,58,58,True,True
1,2026,774239,774239,58,58,True,True
